# 부산 아파트 실거래가 데이터 탐색

## 목적

국토교통부 실거래가 원본 데이터의 구조와 품질을 확인하고,
전처리가 필요한 항목을 파악한다.

이번 단계에서는 데이터를 수정하지 않고 다음 내용을 확인한다.

- 원본 파일 구조
- 데이터 행/열 개수
- 컬럼 구성
- 데이터 타입
- 결측치
- 중복 데이터
- 기초 통계

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
DATA_PATH = Path("../data/raw/부산_아파트(매매)_실거래가_202508_202607.csv")

df = pd.read_csv(
    DATA_PATH,
    encoding="cp949",
    skiprows=15
)

df.shape

(38163, 20)

## 등기 완료 후 동 정보 미기재 원인 확인

이전 분석에서 `등기일자`가 미기재된 7,041건은 모두 `동` 정보도 미기재되어 있음을 확인했다.

그러나 등기일자가 존재함에도 `동` 정보가 `-`로 표시된 거래가 2,944건 존재했다.

따라서 해당 데이터가 특정 지역이나 단지에 집중되어 있는지 확인하여
동 정보 미기재의 추가 원인을 탐색한다.

In [3]:
registered_dong_missing = df[
    (df["등기일자"] != "-") &
    (df["동"] == "-")
]

len(registered_dong_missing)

2944

In [4]:
registered_dong_missing[
    ["시군구", "단지명", "도로명", "계약년월", "등기일자"]
].head(20)

,시군구,단지명,도로명,계약년월,등기일자
30,부산광역시 남구 감만동,하얏트,전선등로 31,202607,26.08.10
34,부산광역시 남구 문현동,무학프라자,수영로 74-5,202607,26.07.31
66,부산광역시 사하구 괴정동,괴정동그린시티,사하로 169,202607,26.07.31
73,부산광역시 금정구 남산동,현대그레이스,벅구산로 25-5,202607,26.08.03
120,부산광역시 부산진구 전포동,경원프라우드-2,전포대로 163,202607,26.07.30
135,부산광역시 남구 문현동,시티프라자,수영로 26,202607,26.08.06
197,부산광역시 사하구 당리동,옥돌,괴정로 89,202607,26.07.29
255,부산광역시 남구 용당동,용당대광,유엔평화로126번길 19,202607,26.07.30
258,부산광역시 북구 구포동,어반펠리체,낙동대로1766번길 17,202607,26.08.14
278,부산광역시 중구 보수동2가,보수2차봄여름가을겨울,보수대로118번길 49,202607,26.07.28


In [5]:
registered_dong_missing["단지명"].value_counts().head(20)

단지명
온천동유림노르웨이숲       26
시티프라자            22
SK허브올리브          21
타워베르빌            21
서면대우디오빌2         19
극동정림             19
한성기린             19
서면베르빌2           18
롯데골드로즈           18
윤성리버티            16
장전경보             16
삼익               16
연산센터빌            15
협성스카이라인80        14
한솔폴라리스           14
한일유앤아이(890-1)    14
래미안포레스티지         14
아이뷰파크            13
사상역포르투나더테라스      13
파크블루11차          13
Name: count, dtype: int64

### 등기 완료 후 동 정보 미기재 원인 가설

등기일자가 존재함에도 `동` 정보가 없는 거래가 2,944건 존재했다.

해당 거래의 일부를 확인한 결과,
단일 건물 형태로 보이는 소규모 아파트가 다수 포함되어 있었다.

따라서 실제 결측이라기보다
건물 자체에 별도의 동 구분이 없어 `동` 정보가 제공되지 않는 경우일 가능성을 확인한다.

이를 위해 동일 단지의 다른 거래에서는 동 정보가 존재하는지 비교한다.

In [6]:
dong_missing_complexes = registered_dong_missing["단지명"].unique()

df[
    df["단지명"].isin(dong_missing_complexes)
].groupby("단지명")["동"].apply(
    lambda x: (x != "-").sum()
).sort_values().head(30)

단지명
희재하이빌        0
가화만사성4차      0
강일프라임빌       0
강진노블레스       0
강진융프라우       0
개금동포르투나      0
개금블루스카이      0
개금역대상웰리움     0
개금코아팰리스      0
개나리          0
거성           0
거성경원         0
건오힐타워        0
협성센트로3차      0
경동윈츠빌        0
흑교           0
(207-3)      0
화송빌라         0
화신거화         0
화신거화2        0
협성스카이라인80    0
화원레샹스        0
협진태양맨션A동     0
화인아트         0
화지빌라트        0
화천승학         0
황중베르빌        0
효림시티빌        0
효성           0
휴그린          0
Name: 동, dtype: int64

### 동일 단지의 동 정보 존재 여부 확인

동 정보가 미기재된 단지들의 다른 거래를 확인하여,
해당 단지에서 동 정보가 한 번이라도 기록된 적이 있는지 확인한다.

동 정보가 전혀 기록되지 않은 단지가 대부분이라면,
단일 건물 형태이거나 별도의 동 구분이 없는 단지일 가능성을 고려할 수 있다.

In [7]:
dong_presence_by_complex = (
    df[df["단지명"].isin(dong_missing_complexes)]
    .groupby("단지명")["동"]
    .apply(lambda x: (x != "-").sum())
)

(dong_presence_by_complex > 0).value_counts()

동
False    1003
True       58
Name: count, dtype: int64

In [8]:
dong_presence_by_complex[
    dong_presence_by_complex > 0
].sort_values(ascending=False).head(20)

단지명
현대         184
삼성         109
경동          87
동원          66
협진태양        48
삼익          32
우성          31
화인          27
신동아         23
삼한사랑채       23
대원          19
한성기린        17
금강          17
비룡벨로스텔라     16
금호          13
로얄          13
일동지에닌       12
한성          11
대림비치        10
허브팰리스        8
Name: 동, dtype: int64

### 단지 식별 기준 보완

단지명 기준으로 동 정보 존재 여부를 확인한 결과,
1,061개 단지명 중 1,003개에서는 동 정보가 한 번도 확인되지 않았다.

다만 `현대`, `삼성`, `삼익` 등 동일한 단지명이 여러 지역에서
사용될 수 있으므로 단지명만으로 서로 다른 아파트를 동일 단지로
판단할 가능성이 있다.

따라서 `시군구`, `단지명`, `도로명`을 함께 사용하여
단지를 보다 구체적으로 구분한 뒤 다시 확인한다.

In [9]:
missing_complex_keys = (
    registered_dong_missing[
        ["시군구", "단지명", "도로명"]
    ]
    .drop_duplicates()
)

matching_complex_trades = df.merge(
    missing_complex_keys,
    on=["시군구", "단지명", "도로명"],
    how="inner"
)

dong_presence_by_complex = (
    matching_complex_trades
    .groupby(["시군구", "단지명", "도로명"])["동"]
    .apply(lambda x: (x != "-").sum())
)

(dong_presence_by_complex > 0).value_counts()

동
False    1109
Name: count, dtype: int64

### 동 정보 미기재 원인 분석 결과

단지명만으로 단지를 구분할 경우 동일한 이름의 서로 다른 아파트가
하나의 단지로 집계될 가능성이 있어,
`시군구`, `단지명`, `도로명`을 함께 사용하여 단지를 다시 구분했다.

그 결과 등기일자가 존재하지만 `동` 정보가 미기재된 거래가 있는
1,109개 단지 조합에서는 현재 분석 데이터 전체에서
동 정보가 기록된 거래가 한 건도 확인되지 않았다.

따라서 해당 `동` 미기재 값은 개별 거래의 단순한 데이터 누락이라기보다,
해당 건물에서 별도의 동 정보가 제공되지 않는 구조적 특성일 가능성이 높은 것으로 판단했다.

다만 현재 데이터만으로 건물의 실제 구조를 직접 확인한 것은 아니므로,
'단일 동 건물'이라고 단정하지 않고 '동 정보 미제공 또는 동 구분 없음'으로 해석한다.

따라서 `동`이 `-`인 데이터를 이유로 거래 행을 삭제하지 않는다.

## 실제 결측치(NaN) 확인

원본 데이터에서는 값이 없는 항목이 주로 `-`로 표현되어 있었으며,
해당 값의 의미를 컬럼별로 분석했다.

이번에는 Pandas가 실제 결측값(`NaN`)으로 인식하고 있는 데이터가
별도로 존재하는지 확인한다.

In [10]:
df.isna().sum()

NO          0
시군구         0
번지          0
본번          0
부번          0
단지명         0
전용면적(㎡)     0
계약년월        0
계약일         0
거래금액(만원)    0
동           0
층           0
매수자         0
매도자         0
건축년도        0
도로명         0
해제사유발생일     0
거래유형        0
중개사소재지      0
등기일자        0
dtype: int64

In [11]:
df.isna().sum().sum()

np.int64(0)

In [12]:
df.duplicated().sum()

np.int64(0)

### 실제 결측치 확인 결과

`df.isna()`를 통해 Pandas가 실제 결측값으로 인식하는 데이터를 확인한 결과,
모든 컬럼에서 `NaN`은 발견되지 않았다.

따라서 이 데이터에서는 값이 없는 항목이 `NaN`보다는
주로 `-` 문자열로 표현되어 있음을 확인했다.

## 중복 데이터 확인

동일한 거래 데이터가 중복으로 포함되어 있는지 확인한다.

전체 컬럼을 기준으로 확인한 결과 중복 행은 발견되지 않았지만,
`NO` 컬럼은 각 행에 부여된 순번이므로 실제 거래 내용의 중복 여부를 확인하기 위해
`NO` 컬럼을 제외하고 다시 검사한다.

In [13]:
df.drop(columns=["NO"]).duplicated().sum()

np.int64(145)

### 중복 데이터 1차 확인 결과

전체 컬럼을 기준으로 중복 여부를 확인했을 때는 중복 행이 발견되지 않았다.

그러나 각 행의 순번 역할을 하는 `NO` 컬럼을 제외하고 다시 확인한 결과,
145건의 중복 후보가 발견되었다.

동일한 거래정보가 실제 중복 저장된 것인지,
동일 조건으로 여러 건의 거래가 발생한 것인지 확인한 후 처리 여부를 결정한다.

duplicated()는 뒤쪽 한 행만 중복으로 잡는데,
keep=False로 해야 둘 다 보여줌

In [14]:
duplicate_rows = df[
    df.drop(columns=["NO"]).duplicated(keep=False)
]

duplicate_rows.head(20)

,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),동,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자
1237,1238,부산광역시 연제구 연산동,2382,2382,0,힐스테이트연산,39.9266,202607,15,"25,800",-,3,개인,법인,2021,봉수로 2,-,직거래,-,-
1238,1239,부산광역시 연제구 연산동,2382,2382,0,힐스테이트연산,39.9266,202607,15,"25,800",-,3,개인,법인,2021,봉수로 2,-,직거래,-,-
1665,1666,부산광역시 연제구 거제동,1519,1519,0,거제센트럴자이,59.9635,202607,10,"67,500",-,13,개인,개인,2018,법원북로 93,-,중개거래,부산 연제구,-
1666,1667,부산광역시 연제구 거제동,1519,1519,0,거제센트럴자이,59.9635,202607,10,"67,500",-,13,개인,개인,2018,법원북로 93,-,중개거래,부산 연제구,-
3235,3236,부산광역시 동구 수정동,389-20,389,20,북항에코하임센트럴뷰,48.1760,202606,23,"29,648",주건축물제1동,7,개인,법인,2023,초량상로 138,-,직거래,-,26.06.24
3236,3237,부산광역시 동구 수정동,389-20,389,20,북항에코하임센트럴뷰,48.1760,202606,23,"29,648",주건축물제1동,7,개인,법인,2023,초량상로 138,-,직거래,-,26.06.24
3239,3240,부산광역시 동구 수정동,389-20,389,20,북항에코하임센트럴뷰,48.1760,202606,23,"30,032",주건축물제1동,11,개인,법인,2023,초량상로 138,-,직거래,-,26.06.24
3240,3241,부산광역시 동구 수정동,389-20,389,20,북항에코하임센트럴뷰,48.1760,202606,23,"30,032",주건축물제1동,11,개인,법인,2023,초량상로 138,-,직거래,-,26.06.24
3246,3247,부산광역시 동구 수정동,389-20,389,20,북항에코하임센트럴뷰,48.1760,202606,23,"30,533",주건축물제1동,16,개인,법인,2023,초량상로 138,-,직거래,-,26.06.24
3247,3248,부산광역시 동구 수정동,389-20,389,20,북항에코하임센트럴뷰,48.1760,202606,23,"30,533",주건축물제1동,16,개인,법인,2023,초량상로 138,-,직거래,-,26.06.24


In [15]:
duplicate_rows.shape

(237, 20)

### 중복 데이터 분석 결과

`NO` 컬럼을 제외하고 중복 여부를 확인한 결과,
145건의 추가 중복 행이 확인되었다.

중복 원본까지 포함하면 총 237개의 행이 92개의 동일 데이터 그룹으로 구성되어 있었다.

일부 데이터를 직접 확인한 결과,
계약일, 단지명, 전용면적, 층, 거래금액 등 제공되는 모든 거래 정보가
동일한 거래가 여러 건 존재했다.

그러나 현재 공개 데이터에는 개별 세대의 호수 등
각 거래를 완전히 식별할 수 있는 정보가 포함되어 있지 않다.

따라서 동일 아파트에서 같은 날짜에 동일한 조건으로
서로 다른 세대가 거래된 경우도 현재 데이터에서는 동일 행으로 나타날 수 있다.

이에 따라 해당 데이터를 단순 중복 오류로 판단하여 삭제하지 않고,
실제 거래일 가능성이 있는 데이터로 유지한다.

In [16]:
duplicate_cols = df.columns.drop("NO")

duplicate_group_sizes = (
    df.groupby(list(duplicate_cols))
      .size()
)

duplicate_group_sizes[duplicate_group_sizes > 1].value_counts().sort_index()

2    66
3    10
4     8
5     7
8     1
Name: count, dtype: int64

추가로 중복 그룹별 반복 횟수를 확인한 결과,

- 2회 반복: 66개 그룹
- 3회 반복: 10개 그룹
- 4회 반복: 8개 그룹
- 5회 반복: 7개 그룹
- 8회 반복: 1개 그룹

총 92개의 동일 데이터 그룹에서 237건의 거래가 확인되었다.

공개 데이터만으로는 개별 세대를 완전히 식별할 수 없으므로,
이들을 실제 중복 데이터라고 확정할 수 없다.
따라서 탐색 단계에서는 해당 행을 삭제하지 않는다.

## 자료형 변환 필요 항목 확인

원본 데이터의 각 컬럼이 분석에 적합한 자료형으로 저장되어 있는지 확인한다.

특히 `거래금액(만원)`은 금액을 나타내는 수치 데이터임에도
문자열(`str`)로 저장되어 있어 원인을 확인하고 숫자형 변환 가능 여부를 점검한다.

In [17]:
df["거래금액(만원)"].head(20)

0     39,000
1     33,000
2      8,200
3     35,000
4     43,800
5     34,400
6     20,900
7     39,000
8     84,000
9     34,000
10    22,100
11    21,300
12    59,000
13    39,000
14    84,200
15    45,000
16    57,500
17    37,000
18    15,700
19    40,000
Name: 거래금액(만원), dtype: str

In [18]:
df["거래금액(만원)"].sample(20, random_state=42)

37439    41,000
11898    85,000
24628     5,950
26980    70,000
35203    42,000
26895    28,000
12539    25,700
19161    74,100
29597    64,000
13906    22,000
29110    27,100
34352    12,500
1704     38,000
8500     38,500
21562    62,000
584      51,500
13118    15,500
37581    42,500
18560    36,500
19635    60,000
Name: 거래금액(만원), dtype: str

In [19]:
price_cleaned = df["거래금액(만원)"].str.replace(",", "", regex=False)

pd.to_numeric(price_cleaned, errors="coerce").isna().sum()

np.int64(0)

### 거래금액 자료형 확인 결과

`거래금액(만원)`은 금액 데이터임에도 문자열(`str`)로 저장되어 있었다.

실제 값을 확인한 결과 `39,000`, `8,200`과 같이
천 단위 구분을 위한 쉼표(`,`)가 포함되어 있었다.

쉼표를 제거한 뒤 숫자형 변환 가능 여부를 확인한 결과,
전체 38,163건에서 변환 불가능한 값은 0건이었다.

따라서 전처리 단계에서는 쉼표를 제거한 뒤
`거래금액(만원)`을 정수형으로 변환한다.

### 날짜 데이터 형식 확인

계약 날짜가 `계약년월`과 `계약일` 두 컬럼으로 분리되어 있으며,
날짜형(`datetime`)이 아닌 정수형으로 저장되어 있다.

향후 월별·기간별 거래 분석을 위해
두 컬럼을 결합하여 하나의 계약일자 컬럼으로 변환할 수 있는지 확인한다.

In [20]:
df[["계약년월", "계약일"]].head(20)

,계약년월,계약일
0,202607,31
1,202607,31
2,202607,31
3,202607,31
4,202607,31
5,202607,31
6,202607,31
7,202607,31
8,202607,31
9,202607,31


In [21]:
contract_date_test = pd.to_datetime(
    df["계약년월"].astype(str)
    + df["계약일"].astype(str).str.zfill(2),
    format="%Y%m%d",
    errors="coerce"
)

contract_date_test.head(20)

0    2026-07-31
1    2026-07-31
2    2026-07-31
3    2026-07-31
4    2026-07-31
5    2026-07-31
6    2026-07-31
7    2026-07-31
8    2026-07-31
9    2026-07-31
10   2026-07-31
11   2026-07-31
12   2026-07-31
13   2026-07-31
14   2026-07-31
15   2026-07-31
16   2026-07-31
17   2026-07-31
18   2026-07-31
19   2026-07-31
dtype: datetime64[us]

In [22]:
contract_date_test.isna().sum()

np.int64(0)

### 계약일자 변환 확인 결과

`계약년월`과 `계약일`을 결합하여 날짜형(`datetime`)으로 변환한 결과,
전체 38,163건에서 변환 실패 데이터는 발생하지 않았다.

따라서 이후 전처리 단계에서는
두 컬럼을 결합하여 `계약일자` 컬럼을 생성한다.

### 해제사유발생일 및 등기일자 형식 확인

`해제사유발생일`과 `등기일자`는 날짜 정보를 나타내지만
문자열(`str`)로 저장되어 있으며, 값이 없는 경우 `-`로 표시되어 있다.

`-`를 날짜 없음으로 처리했을 때
나머지 값들이 정상적으로 날짜형으로 변환 가능한지 확인한다.

In [23]:
cancel_date_test = pd.to_datetime(
    df["해제사유발생일"].replace("-", pd.NA),
    format="%Y%m%d",
    errors="coerce"
)

registration_date_test = pd.to_datetime(
    df["등기일자"].replace("-", pd.NA),
    errors="coerce"
)

C:\Users\it\AppData\Local\Temp\ipykernel_31476\625625915.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  registration_date_test = pd.to_datetime(


In [24]:
cancel_date_test.head(20)

0    NaT
1    NaT
2    NaT
3    NaT
4    NaT
5    NaT
6    NaT
7    NaT
8    NaT
9    NaT
10   NaT
11   NaT
12   NaT
13   NaT
14   NaT
15   NaT
16   NaT
17   NaT
18   NaT
19   NaT
Name: 해제사유발생일, dtype: datetime64[us]

In [25]:
registration_date_test.head(20)

0           NaT
1           NaT
2           NaT
3           NaT
4           NaT
5           NaT
6    2011-08-26
7    2031-07-26
8           NaT
9           NaT
10          NaT
11          NaT
12          NaT
13   2004-08-26
14          NaT
15          NaT
16          NaT
17          NaT
18          NaT
19          NaT
Name: 등기일자, dtype: datetime64[us]

In [26]:
print("해제사유발생일 원래 '-' 개수:", (df["해제사유발생일"] == "-").sum())
print("해제사유발생일 변환 후 NaT:", cancel_date_test.isna().sum())

print("등기일자 원래 '-' 개수:", (df["등기일자"] == "-").sum())
print("등기일자 변환 후 NaT:", registration_date_test.isna().sum())

해제사유발생일 원래 '-' 개수: 35940
해제사유발생일 변환 후 NaT: 35940
등기일자 원래 '-' 개수: 7041
등기일자 변환 후 NaT: 7041


### 등기일자 날짜 변환 오류 확인

`등기일자`를 날짜형으로 시험 변환한 결과,
결측값 개수는 원본의 `-` 개수와 동일하게 나타났다.

그러나 실제 변환 결과를 확인했을 때
`26.08.10`과 같은 값이 `2011-08-26` 등 잘못된 날짜로 변환되는 문제가 발견되었다.

Pandas가 날짜 형식을 자동으로 추론하면서
`YY.MM.DD` 형식을 정확하게 해석하지 못한 것으로 판단했다.

따라서 날짜 데이터는 자동 추론에 의존하지 않고
원본 형식에 맞는 `format`을 명시하여 다시 변환한다.

In [27]:
registration_date_test = pd.to_datetime(
    df["등기일자"].replace("-", pd.NA),
    format="%y.%m.%d",
    errors="coerce"
)

registration_date_test.head(20)

0           NaT
1           NaT
2           NaT
3           NaT
4           NaT
5           NaT
6    2026-08-11
7    2026-07-31
8           NaT
9           NaT
10          NaT
11          NaT
12          NaT
13   2026-08-04
14          NaT
15          NaT
16          NaT
17          NaT
18          NaT
19          NaT
Name: 등기일자, dtype: datetime64[us]

In [28]:
print("등기일자 원래 '-' 개수:", (df["등기일자"] == "-").sum())
print("등기일자 변환 후 NaT:", registration_date_test.isna().sum())

등기일자 원래 '-' 개수: 7041
등기일자 변환 후 NaT: 7041


In [29]:
registration_date_test.dropna().min(), registration_date_test.dropna().max()

(Timestamp('2025-08-01 00:00:00'), Timestamp('2026-08-19 00:00:00'))

In [30]:
pd.DataFrame({
    "원본": df["등기일자"],
    "변환결과": registration_date_test
}).query("원본 != '-'").head(20)

,원본,변환결과
6,26.08.11,2026-08-11
7,26.07.31,2026-07-31
13,26.08.04,2026-08-04
30,26.08.10,2026-08-10
34,26.07.31,2026-07-31
66,26.07.31,2026-07-31
68,26.07.31,2026-07-31
73,26.08.03,2026-08-03
81,26.08.06,2026-08-06
82,26.08.05,2026-08-05


### 날짜형 변환 확인 결과

날짜 관련 컬럼의 원본 형식을 확인하고 날짜형(`datetime`) 변환 가능 여부를 검증했다.

- `계약년월` + `계약일` → `%Y%m%d` 형식으로 결합 가능
- `해제사유발생일` → `%Y%m%d` 형식
- `등기일자` → `%y.%m.%d` 형식

`등기일자`는 처음에 날짜 형식을 지정하지 않고 변환했을 때
`26.08.11`이 잘못된 날짜로 해석되는 문제가 발생했다.

원본 형식을 확인한 후 `format="%y.%m.%d"`를 명시하여 다시 변환했고,
원본 값과 변환 결과가 정상적으로 일치하는 것을 확인했다.

또한 `등기일자`의 원본 `-` 7,041건과
변환 후 `NaT` 7,041건이 일치하여,
`-` 이외의 날짜값은 모두 정상적으로 변환 가능함을 확인했다.

## 기초 통계 및 이상치 후보 확인

거래금액, 전용면적, 층, 건축년도 등 주요 수치형 데이터의
최솟값, 최댓값, 평균, 중앙값 등을 확인하여
비정상적이거나 추가 확인이 필요한 값을 탐색한다.

이상치로 보이는 값이 발견되더라도 실제 거래일 가능성이 있으므로
단순히 삭제하지 않고 원본 데이터를 확인한 후 판단한다.

In [31]:
price_numeric = pd.to_numeric(
    df["거래금액(만원)"].str.replace(",", "", regex=False)
)

In [32]:
numeric_summary = df[
    ["전용면적(㎡)", "층", "건축년도"]
].describe()

numeric_summary

,전용면적(㎡),층,건축년도
count,38163.000000,38163.000000,38163.000000
mean,77.177830,12.665199,2007.693761
std,24.227596,8.979382,11.772637
min,12.764400,1.000000,1962.000000
25%,59.925000,6.000000,1998.000000
50%,82.750000,11.000000,2008.000000
75%,84.970000,18.000000,2018.000000
max,244.849000,82.000000,2026.000000


In [33]:
price_numeric.describe()

count     38163.000000
mean      45004.650971
std       33994.016277
min        1500.000000
25%       22500.000000
50%       37500.000000
75%       58500.000000
max      570000.000000
Name: 거래금액(만원), dtype: float64

### 극단값 확인

기초 통계에서 전용면적, 층, 건축년도 등의 최솟값과 최댓값을 확인했다.

극단적인 값이 존재하더라도 실제 아파트 거래일 수 있으므로
단순히 이상치로 제거하지 않고 해당 거래의 원본 정보를 확인한다.

In [34]:
df.loc[
    df["전용면적(㎡)"].idxmax(),
    ["시군구", "단지명", "전용면적(㎡)", "층", "거래금액(만원)", "건축년도"]
]

시군구                부산광역시 남구 용호동
단지명         엘지메트로시티4-1(201-214)
전용면적(㎡)                 244.849
층                             9
거래금액(만원)                 94,000
건축년도                       2003
Name: 2036, dtype: object

In [35]:
df.loc[
    df["층"].idxmax(),
    ["시군구", "단지명", "전용면적(㎡)", "층", "거래금액(만원)", "건축년도"]
]

시군구         부산광역시 해운대구 중동
단지명                   엘시티
전용면적(㎡)          161.9826
층                      82
거래금액(만원)          400,000
건축년도                 2019
Name: 13690, dtype: object

In [36]:
df.loc[
    df["건축년도"].idxmin(),
    ["시군구", "단지명", "전용면적(㎡)", "층", "거래금액(만원)", "건축년도"]
]

시군구         부산광역시 동구 좌천동
단지명          좌천시민(737-1)
전용면적(㎡)             36.4
층                      4
거래금액(만원)           3,000
건축년도                1962
Name: 6520, dtype: object

### 주요 수치형 데이터 기초 통계 확인 결과

전용면적, 층, 건축년도 및 거래금액의 기초 통계를 확인했다.

주요 극단값은 다음과 같다.

- 최대 전용면적: 244.849㎡
- 최고층: 82층
- 가장 오래된 건축년도: 1962년
- 최소 거래금액: 1,500만원
- 최대 거래금액: 570,000만원

극단값에 해당하는 일부 거래를 직접 확인한 결과,
초고층 아파트나 대형 평형 등 실제 거래로 설명 가능한 데이터가 존재했다.

따라서 최솟값·최댓값만을 기준으로 이상치를 제거하지 않고,
각 거래의 실제 내용을 추가 확인한 후 처리 여부를 판단한다.

또한 거래금액은 평균 약 45,005만원에 비해 중앙값이 37,500만원으로 낮게 나타나,
일부 고가 거래가 평균에 영향을 주고 있을 가능성이 있다.

### 거래금액 극단값 확인

거래금액의 최소값과 최대값이 실제 거래인지 확인하기 위해
해당 행의 단지명, 면적, 층, 지역 등의 정보를 직접 확인한다.

In [37]:
df.loc[
    price_numeric.idxmax(),
    ["시군구", "단지명", "전용면적(㎡)", "층",
     "거래금액(만원)", "건축년도", "계약년월", "계약일"]
]

시군구         부산광역시 해운대구 우동
단지명            해운대 I PARK
전용면적(㎡)            205.34
층                      71
거래금액(만원)          570,000
건축년도                 2011
계약년월               202511
계약일                     6
Name: 27799, dtype: object

In [38]:
df.loc[
    price_numeric.idxmin(),
    ["시군구", "단지명", "전용면적(㎡)", "층",
     "거래금액(만원)", "건축년도", "계약년월", "계약일"]
]

시군구         부산광역시 동구 좌천동
단지명          좌천시민(741-1)
전용면적(㎡)             36.4
층                      5
거래금액(만원)           1,500
건축년도                1962
계약년월              202512
계약일                   23
Name: 21980, dtype: object

### 거래금액 극단값 확인 결과

거래금액의 최댓값과 최솟값에 해당하는 거래를 직접 확인했다.

- 최고가: 해운대 I PARK, 205.34㎡, 71층, 570,000만원
- 최저가: 좌천시민(741-1), 36.4㎡, 5층, 1,500만원

최고가 거래는 대형 면적과 초고층이라는 특성이 함께 확인되어
단순 오류값으로 보기 어렵다.

최저가 거래 역시 소형 면적과 1962년 건축이라는 특성이 있으나,
다른 거래에 비해 지나치게 낮은 가격인지 확인하기 위해
동일 단지의 거래가격 분포를 추가로 비교한다.

In [39]:
min_price_row = df.loc[price_numeric.idxmin()]

same_complex = df[
    (df["시군구"] == min_price_row["시군구"]) &
    (df["단지명"] == min_price_row["단지명"]) &
    (df["도로명"] == min_price_row["도로명"])
].copy()

same_complex_prices = pd.to_numeric(
    same_complex["거래금액(만원)"].str.replace(",", "", regex=False)
)

same_complex_prices.describe()

count       4.000000
mean     1925.000000
std       567.890835
min      1500.000000
25%      1500.000000
50%      1750.000000
75%      2175.000000
max      2700.000000
Name: 거래금액(만원), dtype: float64

In [40]:
same_complex.assign(
    거래금액_숫자=same_complex_prices
)[
    ["계약년월", "계약일", "전용면적(㎡)", "층", "거래금액_숫자"]
].sort_values("거래금액_숫자")

,계약년월,계약일,전용면적(㎡),층,거래금액_숫자
21980,202512,23,36.4,5,1500
37405,202508,11,36.4,1,1500
24216,202512,3,36.4,5,2000
16100,202602,11,36.4,2,2700


### 최저 거래금액 검증 결과

전체 데이터의 최저 거래금액인 1,500만원이 실제 이상치인지 확인하기 위해
동일 단지의 다른 거래와 비교했다.

동일 단지·동일 전용면적(36.4㎡)의 거래가격은 다음과 같이 확인되었다.

- 1,500만원
- 1,500만원
- 2,000만원
- 2,700만원

최저가인 1,500만원과 비슷한 수준의 거래가 여러 건 존재하므로,
해당 값은 데이터 입력 오류라기보다 실제 저가 거래일 가능성이 높다고 판단했다.

따라서 단순히 전체 데이터의 최솟값이라는 이유로 제거하지 않는다.

## IQR을 이용한 거래금액 이상치 후보 탐색

최솟값과 최댓값만으로는 전체 데이터의 분포를 충분히 확인하기 어렵다.

따라서 IQR(Interquartile Range)을 이용하여
통계적으로 일반적인 가격 범위에서 크게 벗어나는 거래를 탐색한다.

IQR 기준으로 탐지된 값은 실제 고가·저가 거래일 수 있으므로
자동으로 삭제하지 않고 이상치 후보로만 분류한다.

In [41]:
q1 = price_numeric.quantile(0.25)
q3 = price_numeric.quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("하한값:", lower_bound)
print("상한값:", upper_bound)

Q1: 22500.0
Q3: 58500.0
IQR: 36000.0
하한값: -31500.0
상한값: 112500.0


In [42]:
price_outliers = df[
    (price_numeric < lower_bound) |
    (price_numeric > upper_bound)
]

price_outliers.shape

(1351, 20)

### IQR 이상치 탐색 결과

거래금액의 IQR을 계산한 결과 다음과 같은 기준이 확인되었다.

- Q1: 22,500만원
- Q3: 58,500만원
- IQR: 36,000만원
- 하한값: -31,500만원
- 상한값: 112,500만원

IQR 기준으로 총 1,351건의 이상치 후보가 탐지되었다.

하한값은 음수이므로 실제 저가 거래가 하한 이상치로 분류될 가능성은 없으며,
이번 데이터에서는 주로 112,500만원을 초과하는 고가 거래가 이상치 후보에 포함된다.

다만 부산 전체 아파트를 하나의 가격 분포로 비교하고 있으므로,
지역·단지·면적 등에 따른 정상적인 고가 거래도 이상치로 탐지될 수 있다.

따라서 IQR 이상치 후보를 자동으로 삭제하지 않고
어떤 지역과 단지의 거래가 포함되어 있는지 추가로 확인한다.

In [43]:
price_outliers_view = price_outliers.copy()

price_outliers_view["거래금액_숫자"] = price_numeric.loc[
    price_outliers_view.index
]

price_outliers_view[
    ["시군구", "단지명", "전용면적(㎡)", "층", "거래금액_숫자"]
].sort_values(
    "거래금액_숫자",
    ascending=False
).head(20)

,시군구,단지명,전용면적(㎡),층,거래금액_숫자
27811,부산광역시 해운대구 우동,해운대 I PARK,205.3400,71,570000
27799,부산광역시 해운대구 우동,해운대 I PARK,205.3400,71,570000
29883,부산광역시 해운대구 우동,해운대두산위브더제니스,209.8332,72,470000
5652,부산광역시 해운대구 중동,엘시티,186.0063,43,460000
27388,부산광역시 해운대구 중동,엘시티,186.0063,49,450000
36112,부산광역시 해운대구 중동,엘시티,186.0063,46,450000
21888,부산광역시 해운대구 중동,엘시티,186.0063,41,449000
27818,부산광역시 해운대구 중동,엘시티,186.0063,49,437000
4119,부산광역시 해운대구 우동,해운대두산위브더제니스,159.5413,75,418000
15462,부산광역시 해운대구 중동,엘시티,186.0063,77,403000


In [44]:
price_outliers_view.assign(
    구=price_outliers_view["시군구"].str.split().str[1]
)["구"].value_counts()

구
해운대구    593
수영구     359
남구      202
동래구      89
연제구      83
금정구      13
부산진구      5
서구        4
강서구       2
북구        1
Name: count, dtype: int64

### IQR 이상치 후보 분석 결과

부산 전체 아파트 거래금액을 기준으로 IQR을 적용한 결과
1,351건의 이상치 후보가 탐지되었다.

고가 순으로 실제 거래를 확인한 결과,
해운대 I PARK, 엘시티, 해운대두산위브더제니스 등
고가 아파트의 정상 거래가 다수 포함되어 있었다.

또한 이상치 후보는 다음 지역에 집중되는 경향을 보였다.

- 해운대구: 593건
- 수영구: 359건
- 남구: 202건
- 동래구: 89건
- 연제구: 83건

따라서 부산 전체를 하나의 가격 분포로 비교할 경우
지역별 가격 수준 차이로 인해 정상적인 고가 거래도
통계적 이상치로 분류될 수 있다고 판단했다.

이에 따라 전체 데이터 기준 IQR 결과를
삭제 기준으로 사용하지 않는다.

### 지역별 이상치 후보 비율 확인

지역별 거래량 차이의 영향을 고려하기 위해,
각 구의 전체 거래 중 IQR 이상치 후보가 차지하는 비율을 확인한다.

In [45]:
df_with_price = df.copy()

df_with_price["거래금액_숫자"] = price_numeric
df_with_price["구"] = df_with_price["시군구"].str.split().str[1]

total_by_gu = df_with_price.groupby("구").size()

outlier_by_gu = (
    price_outliers_view
    .assign(구=price_outliers_view["시군구"].str.split().str[1])
    .groupby("구")
    .size()
)

outlier_rate_by_gu = (
    (outlier_by_gu / total_by_gu * 100)
    .fillna(0)
    .sort_values(ascending=False)
)

outlier_rate_by_gu

구
수영구     16.142086
해운대구    11.782237
남구       6.207744
연제구      2.533578
동래구      2.339642
금정구      0.672878
서구       0.420168
강서구      0.128783
부산진구     0.103135
북구       0.030048
동구       0.000000
기장군      0.000000
사상구      0.000000
사하구      0.000000
영도구      0.000000
중구       0.000000
dtype: float64

### 지역별 이상치 후보 비율 분석 결과

부산 전체 거래금액을 기준으로 계산한 IQR 이상치 후보의
지역별 비율을 확인했다.

- 수영구: 약 16.1%
- 해운대구: 약 11.8%
- 남구: 약 6.2%
- 연제구: 약 2.5%
- 동래구: 약 2.3%

수영구와 해운대구처럼 상대적으로 아파트 가격 수준이 높은 지역에서
이상치 후보 비율이 크게 나타났다.

따라서 부산 전체에 동일한 IQR 기준을 적용할 경우,
지역별 가격 수준 차이로 인해 정상적인 고가 거래가
이상치로 분류될 수 있음을 확인했다.

이에 따라 전체 데이터 기준 IQR은 이상치 삭제 기준으로 사용하지 않는다.

### 지역별 IQR 기준 이상치 탐색

전체 부산 데이터를 하나의 가격 분포로 비교하는 방식의 한계를 확인했다.

지역별 가격 수준 차이를 고려하기 위해
각 구의 거래금액 분포를 기준으로 IQR을 다시 계산하여
지역 내부에서도 상대적으로 크게 벗어난 거래를 탐색한다.

In [46]:
def find_outliers_by_gu(group):
    q1 = group["거래금액_숫자"].quantile(0.25)
    q3 = group["거래금액_숫자"].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return group[
        (group["거래금액_숫자"] < lower) |
        (group["거래금액_숫자"] > upper)
    ]

gu_outliers = (
    df_with_price
    .groupby("구", group_keys=False)
    .apply(find_outliers_by_gu)
)

gu_outliers.shape

(1012, 21)

### 지역별 IQR 적용 결과

부산 전체 거래금액을 하나의 분포로 계산했을 때는
1,351건이 이상치 후보로 탐지되었다.

지역별 가격 수준의 차이를 고려하여
각 구별로 IQR 기준을 다시 계산한 결과,
이상치 후보는 1,012건으로 나타났다.

전체 기준보다 이상치 후보가 감소했지만,
두 방식에서 탐지된 거래가 완전히 동일한 것은 아닐 수 있으므로
각 기준에서 어떤 거래가 포함되거나 제외되었는지 비교한다.

In [47]:
overall_outlier_index = set(price_outliers.index)
gu_outlier_index = set(gu_outliers.index)

both = overall_outlier_index & gu_outlier_index
overall_only = overall_outlier_index - gu_outlier_index
gu_only = gu_outlier_index - overall_outlier_index

print("두 기준 모두 이상치:", len(both))
print("전체 기준에서만 이상치:", len(overall_only))
print("구별 기준에서만 이상치:", len(gu_only))

두 기준 모두 이상치: 641
전체 기준에서만 이상치: 710
구별 기준에서만 이상치: 371


### 전체 기준과 지역별 IQR 비교 결과

전체 부산 기준 IQR과 지역별 IQR의 이상치 후보를 비교한 결과 다음과 같이 나타났다.

- 두 기준 모두 이상치: 641건
- 전체 기준에서만 이상치: 710건
- 지역별 기준에서만 이상치: 371건

전체 기준에서만 이상치로 탐지된 거래가 710건 존재한다는 것은,
부산 전체 가격 수준에서는 고가로 보이지만 해당 지역 내부에서는
일반적인 가격 범위에 포함되는 거래가 상당수 존재한다는 의미이다.

반대로 지역별 기준에서만 탐지된 371건은
부산 전체 가격 수준에서는 크게 벗어나지 않지만,
해당 지역의 가격 분포에서는 상대적으로 크게 벗어난 거래이다.

따라서 부동산 거래가격처럼 지역에 따라 가격 수준의 차이가 큰 데이터는
전체 데이터에 하나의 이상치 기준을 적용하는 것보다
지역적 특성을 고려하여 판단하는 것이 더 적절하다고 판단했다.

또한 IQR로 탐지된 값은 실제 고가·저가 거래일 수 있으므로
이상치라는 이유만으로 자동 삭제하지 않는다.

In [48]:
df_with_price.loc[
    list(overall_only),
    ["구", "시군구", "단지명", "전용면적(㎡)", "층", "거래금액_숫자"]
].sort_values(
    "거래금액_숫자",
    ascending=False
).head(20)

,구,시군구,단지명,전용면적(㎡),층,거래금액_숫자
16220,수영구,부산광역시 수영구 남천동,삼익비치,131.27,3,181500
24035,수영구,부산광역시 수영구 남천동,삼익비치,131.27,10,181000
33699,수영구,부산광역시 수영구 남천동,삼익비치,131.27,11,180000
36279,수영구,부산광역시 수영구 남천동,삼익비치,131.27,11,180000
33411,수영구,부산광역시 수영구 남천동,삼익비치,131.27,1,179000
33409,수영구,부산광역시 수영구 남천동,삼익비치,131.27,1,179000
33853,수영구,부산광역시 수영구 남천동,삼익비치,131.27,10,178000
34999,수영구,부산광역시 수영구 남천동,삼익비치,131.27,4,177000
14224,수영구,부산광역시 수영구 남천동,뉴비치,209.76,6,175000
35974,수영구,부산광역시 수영구 남천동,삼익비치,131.27,6,175000


In [49]:
df_with_price.loc[
    list(gu_only),
    ["구", "시군구", "단지명", "전용면적(㎡)", "층", "거래금액_숫자"]
].sort_values(
    "거래금액_숫자",
    ascending=False
).head(20)

,구,시군구,단지명,전용면적(㎡),층,거래금액_숫자
34416,부산진구,부산광역시 부산진구 가야동,가야롯데캐슬골드아너,102.5328,8,112000
10688,부산진구,부산광역시 부산진구 가야동,가야롯데캐슬골드아너,102.5328,33,111500
4711,서구,부산광역시 서구 암남동,힐스테이트이진베이시티아파트,116.1233,55,111000
13655,서구,부산광역시 서구 동대신동2가,삼익,226.6800,2,110000
1253,부산진구,부산광역시 부산진구 부전동,더샵센트럴스타,175.8580,30,110000
2526,북구,부산광역시 북구 화명동,화명롯데캐슬카이저,171.7700,29,108000
21998,서구,부산광역시 서구 암남동,힐스테이트이진베이시티아파트,116.5511,31,107000
37744,북구,부산광역시 북구 화명동,화명롯데캐슬카이저,171.7700,15,106700
29830,부산진구,부산광역시 부산진구 가야동,가야롯데캐슬골드아너,102.5328,5,106500
9391,서구,부산광역시 서구 암남동,힐스테이트이진베이시티아파트,116.5511,17,105000


### 전체 기준과 지역별 기준의 실제 거래 비교

두 IQR 기준에서 다르게 분류된 거래를 직접 확인했다.

전체 부산 기준에서만 이상치로 분류된 거래에는
수영구 남천동의 삼익비치와 같은 고가 아파트 거래가 다수 포함되어 있었다.

해당 거래들은 부산 전체 가격분포에서는 높은 가격이지만,
수영구 내부의 가격 수준을 기준으로 하면 일반적인 범위에 포함되어
지역별 IQR에서는 이상치로 분류되지 않았다.

반대로 지역별 IQR에서만 이상치로 분류된 거래에는
부산진구, 서구, 북구 등의 고가 거래가 포함되어 있었다.

이 거래들은 부산 전체 기준에서는 이상치 상한보다 낮지만,
해당 지역 내부에서는 상대적으로 높은 가격이기 때문에 이상치 후보로 탐지되었다.

이를 통해 부동산 가격 데이터의 이상치를 판단할 때는
전체 데이터에 동일한 기준을 적용하기보다
지역별 가격 수준과 데이터의 특성을 함께 고려해야 함을 확인했다.

따라서 이번 프로젝트에서는 IQR로 탐지된 데이터를 자동 삭제하지 않고,
이상치 탐색 및 검증 결과만 기록한 뒤 원본 거래는 유지한다.